# 16 · Export the inference models to ONNX

Exports the three models of the inference service to ONNX so the service can run them with `onnxruntime`,
without PyTorch. Each export is checked against the PyTorch model.

**ref:** https://docs.ultralytics.com/integrations/onnx

In [ ]:
%pip install -q ultralytics==8.4.46 onnx onnxruntime python-dotenv

## Setup

In [ ]:
import os
import shutil
import sys
from pathlib import Path

import numpy as np
import onnxruntime as ort
import torch

SRC = Path("/kaggle/working/api-services")
if not SRC.exists():   # inputs are read-only; the export writes next to the weights
    shutil.copytree(next(p.parent for p in Path("/kaggle/input").rglob("predictor.py")), SRC)
OUT = Path("/kaggle/working/onnx")
OUT.mkdir(exist_ok=True)
os.environ["MODEL_DIR"] = str(SRC / "weights")
os.environ["MONGO_URI"] = "mongodb://127.0.0.1:1"
os.environ["YOLO_AUTOINSTALL"] = "false"
sys.path.insert(0, str(SRC))

import plate_layers  # noqa: F401
import predictor
from ultralytics import YOLO

print("service code:", SRC)
print(sorted(p.name for p in (SRC / "weights").iterdir()))

## Occupancy (MobileNetV2)

In [ ]:
occ = predictor.load_model("occupancy").eval()
x = torch.rand(1, 3, 224, 224)
torch.onnx.export(occ, x, OUT / "occupancy.onnx", input_names=["image"], output_names=["logits"],
                  dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17, dynamo=False)

## Plate detector and reader

In [ ]:
for name, path, size in [("plate-detector", "plate-detector-yolo26n/best.pt", 640),
                         ("plate-reader", "plate-reader-layout/best.pt", 320)]:
    f = YOLO(str(SRC / "weights" / path)).export(format="onnx", imgsz=size, opset=17, simplify=False, dynamic=False)
    shutil.copy(f, OUT / f"{name}.onnx")
    print(name, "->", OUT / f"{name}.onnx")

## Layout head

The reader's layout head reads the last backbone layer; it is exported as its own small model.

In [ ]:
class Layout(torch.nn.Module):
    def __init__(self, core):
        super().__init__()
        self.core = core

    def forward(self, x):
        return tuple(logits.softmax(1) for logits in self.core.layout_logits(x))


core = YOLO(str(SRC / "weights" / "plate-reader-layout/best.pt")).model.float().eval()
layout = Layout(core).eval()
x = torch.rand(1, 3, 320, 320)
torch.onnx.export(layout, x, OUT / "plate-layout.onnx", input_names=["image"],
                  output_names=["digits", "districts", "letters", "words"], opset_version=17, dynamo=False)

## Check: ONNX gives the same numbers as PyTorch

In [ ]:
def run(name, x):
    s = ort.InferenceSession(str(OUT / f"{name}.onnx"), providers=["CPUExecutionProvider"])
    return s.run(None, {s.get_inputs()[0].name: x.numpy()})


torch.manual_seed(0)
x = torch.rand(4, 3, 224, 224)
with torch.no_grad():
    print("occupancy   max diff", float(np.abs(occ(x).numpy() - run("occupancy", x)[0]).max()))

x = torch.rand(1, 3, 320, 320)
with torch.no_grad():
    want = layout(x)
print("layout head max diff", max(float(np.abs(a.numpy() - b).max()) for a, b in zip(want, run("plate-layout", x))))

for name, path, size in [("plate-detector", "plate-detector-yolo26n/best.pt", 640),
                         ("plate-reader", "plate-reader-layout/best.pt", 320)]:
    m = YOLO(str(SRC / "weights" / path)).model.float().fuse().eval()
    x = torch.rand(1, 3, size, size)
    with torch.no_grad():
        y = m(x)
    y = (y[0] if isinstance(y, (list, tuple)) else y).numpy()
    o = run(name, x)[0]
    print(f"{name:14} output {o.shape}, max diff {float(np.abs(y - o).max()) if y.shape == o.shape else 'shape differs: ' + str(y.shape)}")

for f in sorted(OUT.iterdir()):
    print(f"{f.name:22} {f.stat().st_size / 1e6:.1f} MB")